In [1]:
import chromadb
from PyPDF2 import PdfReader
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain.embeddings import HuggingFaceEmbeddings
import json
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain.retrievers import BM25Retriever, EnsembleRetriever

In [2]:
load_dotenv()

True

In [3]:
def read_pdf(file_path):    
    # Initialize a variable to store all text
    all_text = ""

    with open(file_path, 'rb') as pdf_file:
        # Create a PDF reader object
        pdf_reader = PdfReader(pdf_file)
        
        # Loop through all the pages
        for page_num in range(len(pdf_reader.pages)):
            # Extract text from each page
            page = pdf_reader.pages[page_num]
            all_text += page.extract_text()

    return all_text

In [4]:
llm = ChatGroq(groq_api_key=os.getenv('GROQ_API_KEY'), model_name="llama-3.1-70b-versatile")

In [5]:
prompt_template = ChatPromptTemplate.from_messages(
    [  
        ('system',
         'You are an assistant that are very proficent at highlighting the key qualifications, education, skiils, certificate, and estimating the applicant\'s years of experience. and also determine if they are already graduated or not. Ensure the summary is brief, focusing on relevant qualifications based on the provided CV.'
        ),
         ('human', "this is the curriculum vitae text :\n\n{text}")
    ]
)

generate_summary = prompt_template | llm


In [6]:
def format_cv(file_path):
    all_text = read_pdf(file_path)    
    summary = generate_summary.invoke({"text": all_text}).content
    return summary

In [7]:
documents = []
name = ['salman','calvin','billy','endi','raja']

for index, candidate in enumerate(os.listdir('user_cv_pdf')):
    candidate_file_path = os.path.join('user_cv_pdf', candidate)
    candidate_summary = format_cv(candidate_file_path)
    documents.append(Document(page_content=candidate_summary, metadata={'name': name[index]}))

In [8]:
bm25_dict = {}

for doc in documents:
    bm25_dict[doc.metadata['name']] = doc.page_content

In [9]:
with open('FOR_BM25/cv_summary.json', 'w') as f:
    json.dump(bm25_dict, f, indent=4)

In [29]:
chroma_embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2")

In [19]:
client = chromadb.PersistentClient(path='CV_CHROMADB')

In [74]:
client.delete_collection(name='cv_collection')

ValueError: Collection cv_collection does not exist.

In [219]:
chroma_embedding.model_name

'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

In [75]:
vector_store = Chroma(
    persist_directory='CV_CHROMADB',
    collection_name='cv_collection',
    embedding_function=chroma_embedding,
    collection_metadata={'hnsw:space':'cosine'}
)

In [76]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, ids=uuids)

['b73bf830-ccd1-4e48-950a-98d4b70806b2',
 '403add6b-6e4d-43b6-91cf-ce6b7192a52b',
 'adfb90dc-3f0f-46f6-8df4-1f8f4eb0f5d0',
 'cbef0670-b0a5-4ff1-b0eb-2309cb2e2e1d',
 'dec1d88f-d69f-424b-a9f2-26527793bc71']

In [77]:
vector_store.similarity_search_with_score("more than 3 years experience in machine learning", 3)

[(Document(metadata={'name': 'salman'}, page_content='**Summary of Qualifications:**\n\n* Education: Bachelor of Science in Physics, Universitas Negeri Jakarta (UNJ), Graduated (2022), GPA: 3.24/4.00\n* Skills: \n  - Programming: Python, SQL (BigQuery)\n  - Machine Learning: Regression, Classification, Clustering, Deep Learning (TensorFlow, Keras, PyTorch)\n  - Cloud Platforms: Google Cloud Platform, Microsoft Azure, Oracle Cloud, Alibaba Cloud\n  - Large Language Models (LLMs): LangChain, Fine-Tuning, RAG, Ollama\n  - Computer Vision: YOLO, CNN\n* Certifications: \n  - Oracle Cloud Infrastructure 2024 Generative AI Certified Professional (2024-2026)\n  - Microsoft Certified: Azure Data Scientist Associate (2024-2025)\n  - Microsoft Certified: Azure AI Engineer Associate (2024-2025)\n  - Alibaba Cloud Certified Professional: Cloud Computing (2023-2025)\n  - Alibaba Cloud Certified Professional: Big Data (2023-2025)\n* Experience: Approximately 2 years of experience in AI engineering an

In [55]:
bm25_corpus = [doc.page_content.split() for doc in documents]

In [78]:
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 2

In [57]:
bm25_retriever.invoke("visual basic")

[Document(metadata={'name': 'raja'}, page_content='**Summary of Key Qualifications:**\n\n- **Education:** Currently pursuing Economics at Muhammadiyah Prof. Dr. Hamka University (3.69/4.00 GPA, started in 2020, expected to graduate soon)\n- **Skills:** Proficient in Python, machine learning, deep learning, data science, natural language processing, computer vision, and generative AI. Experienced with TensorFlow, PyTorch, Scikit-Learn, and other relevant frameworks.\n- **Certificates:**\n  - Building Generative AI-Powered Applications with Python (IBM, 2024)\n  - Machine Learning Specialization (DeepLearning.AI & Stanford University, 2023)\n  - Mathematics for Machine Learning (Imperial College London, 2023)\n  - DeepLearning.AI TensorFlow Developer (DeepLearning.AI, 2023)\n  - ArtiﬁciaI Intelligence for GenZ (Studi Independen Kampus Merdeka, 2022)\n- **Experience:** Approximately 1-2 years of experience in machine learning and data science, with internships at PT Salam Pacific Indonesi

In [79]:
vector_retriever = vector_store.as_retriever(search_kwargs={'k': 10})

In [80]:
vector_retriever.invoke("more than 3 years experience in machine learning")

Number of requested results 10 is greater than number of elements in index 5, updating n_results = 5


[Document(metadata={'name': 'salman'}, page_content='**Summary of Qualifications:**\n\n* Education: Bachelor of Science in Physics, Universitas Negeri Jakarta (UNJ), Graduated (2022), GPA: 3.24/4.00\n* Skills: \n  - Programming: Python, SQL (BigQuery)\n  - Machine Learning: Regression, Classification, Clustering, Deep Learning (TensorFlow, Keras, PyTorch)\n  - Cloud Platforms: Google Cloud Platform, Microsoft Azure, Oracle Cloud, Alibaba Cloud\n  - Large Language Models (LLMs): LangChain, Fine-Tuning, RAG, Ollama\n  - Computer Vision: YOLO, CNN\n* Certifications: \n  - Oracle Cloud Infrastructure 2024 Generative AI Certified Professional (2024-2026)\n  - Microsoft Certified: Azure Data Scientist Associate (2024-2025)\n  - Microsoft Certified: Azure AI Engineer Associate (2024-2025)\n  - Alibaba Cloud Certified Professional: Cloud Computing (2023-2025)\n  - Alibaba Cloud Certified Professional: Big Data (2023-2025)\n* Experience: Approximately 2 years of experience in AI engineering and

In [85]:
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever], weights=[0.3, 0.7])

In [86]:
query = 'satuan pengamanan'

In [87]:
tes = ensemble_retriever.get_relevant_documents(query)

Number of requested results 10 is greater than number of elements in index 5, updating n_results = 5


In [84]:
tes

[Document(metadata={'name': 'endi'}, page_content='**Summary of Key Qualifications:**\n\n* **Current Education Status:** Currently in the 7th semester of Information System at Kwik Kian Gie School of Business, expected to graduate soon.\n* **Education:** Information System (2021-Present) at Kwik Kian Gie School of Business, GPA: 3.9, and MIPA (2018-2021) at SMAN 1 JONGGOL.\n* **Years of Experience:** Approximately 1-2 years of experience, mostly in internships and volunteer work.\n* **Skills:**\n\t+ Data Management (Data Analysis, Data Cleansing, ETL, Power BI, Data Visualization, SQL)\n\t+ Web Development (HTML, PHP, CSS, JS)\n\t+ Programming (Python, Java, Visual Basic (.NET))\n\t+ Analysis & Design Skills (Business Process Modeling, Requirements Gathering, Use Case, UML, Flowchart)\n\t+ Soft Skills (Critical Thinking, Analytical Skills, Decision Making, Communication Skills, Teamwork)\n* **Certifications:** Intro to Data Analyst (RevoU) - Jan 2024\n* **Key Projects:** \n\t+ Final Pr

In [25]:
tes = [3,5,5,3,62,4,1,0,9]

my_set = set(tes)


TypeError: 'set' object is not subscriptable

In [1]:
def tinggi_rendah(tes):
    max = None
    min = None
    for x in tes:
        if max is None or x > max:
            max = x
        if min is None or x < min:
            min = x
    return [max, min]

In [2]:
tinggi_rendah([5,2,7,1,8])

[8, 1]
